# 03 — Fine-tuning a Pre-trained Model

**INSEA Innovation Edge · Deep Learning Workshop**

---

In notebook 2 we trained a network **from scratch**. That works for MNIST, but
for real problems (language, vision at scale) training from zero takes days and
millions of examples.

**The modern approach: fine-tuning.**

Start from a model someone else already trained on huge amounts of data, then
**adapt it to your task** with a small dataset. You get great results in minutes.

**Today's task:** sentiment classification — given a movie review, predict
whether it's **positive** or **negative**.

**Our starting point:** `distilbert-base-uncased`, a small pre-trained language
model that already understands English. We'll teach it about sentiment.

In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, pipeline,
)
from datasets import load_dataset
import numpy as np

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Section 1 — Load a Small Dataset

We use the **IMDB** movie review dataset but keep only a small slice so the
notebook runs quickly. In a real project you'd use all 25,000 reviews.

In [ ]:
dataset = load_dataset('imdb')

# Keep a small subset so training runs in a few minutes on CPU
train_small = dataset['train'].shuffle(seed=0).select(range(1000))
test_small  = dataset['test'].shuffle(seed=0).select(range(500))

print(f'Training examples: {len(train_small)}')
print(f'Test examples:     {len(test_small)}')
print()
print('Example review:')
print('-' * 50)
print(train_small[0]['text'][:300], '...')
print(f"\nLabel: {train_small[0]['label']}  (0 = negative, 1 = positive)")

## Section 2 — Tokenize the Text

Neural networks work on numbers, not words. A **tokenizer** converts text into
integer IDs the model understands.

The tokenizer we load here is the *same one* used when DistilBERT was
pre-trained — that's essential, the model only understands its own vocabulary.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_tokenized = train_small.map(tokenize, batched=True)
test_tokenized  = test_small.map(tokenize,  batched=True)

# Quick peek at what a tokenized example looks like
print('Original text:')
print(train_small[0]['text'][:80], '...')
print('\nFirst 15 token IDs:', train_tokenized[0]['input_ids'][:15])
print('Decoded back:', tokenizer.decode(train_tokenized[0]['input_ids'][:15]))

## Section 3 — Load the Pre-trained Model

`AutoModelForSequenceClassification` loads DistilBERT and automatically
attaches a small **classification head** on top (a new `nn.Linear` layer).

- The DistilBERT body is already trained — it knows English
- The classification head is new and random — we'll train it

During fine-tuning, gradients flow through **both** — the whole model adapts
slightly to the new task.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2     # 2 classes: negative, positive
).to(device)

# Count parameters so we see how big this model is compared to our MNIST MLP
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME}')
print(f'Total parameters: {n_params:,}')

## Section 4 — Fine-tune with the Trainer API

Hugging Face's `Trainer` wraps the training loop from notebook 2
(forward → loss → backward → step) into a single `.train()` call.

Everything you already understand is still happening under the hood.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {'accuracy': accuracy}

training_args = TrainingArguments(
    output_dir='./finetune_output',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,                 # fine-tuning uses small LRs
    eval_strategy='epoch',
    logging_steps=25,
    report_to='none',
    save_strategy='no',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

trainer.train()

## Section 5 — Evaluate

How well did our fine-tuned model do on reviews it has never seen?

In [ ]:
results = trainer.evaluate()
print(f"Test accuracy: {results['eval_accuracy'] * 100:.2f}%")

## Section 6 — Try It on New Text

Let's wrap the model in a `pipeline` for easy predictions.

In [ ]:
classifier = pipeline(
    'sentiment-analysis',
    model=model,
    tokenizer=tokenizer,
    device=0 if device == 'cuda' else -1,
)

# LABEL_0 = negative, LABEL_1 = positive
examples = [
    "This movie was absolutely wonderful, I loved every minute of it!",
    "Terrible plot, bad acting, complete waste of two hours.",
    "Not the best film ever made, but I had a pretty good time.",
    "I fell asleep halfway through. So boring.",
]

for text in examples:
    result = classifier(text)[0]
    sentiment = 'POSITIVE' if result['label'] == 'LABEL_1' else 'NEGATIVE'
    print(f"[{sentiment}  {result['score']:.2f}]  {text}")

## Recap

- **Fine-tuning** starts from a pre-trained model and adapts it to a new task
- You need **orders of magnitude less data and time** than training from scratch
- The Hugging Face `Trainer` wraps the loop you wrote in notebook 2
- Use small learning rates (`2e-5`) so you don't destroy the pre-trained weights

## Where to go from here

- Try a larger subset of IMDB — accuracy goes up fast
- Swap the dataset for another classification task (spam, topic, language)
- Explore other model families: BERT, RoBERTa, smaller ones like TinyBERT
- Look into **LoRA** / PEFT for fine-tuning huge models cheaply

---

*INSEA Innovation Edge · Deep Learning Workshop*